[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C01_LLM_Internals_Course/06_kv_cache_inference/06_kv_cache.ipynb)

# 06 · KV Cache 与高效推理

配套讲解：`06_讲解.html`。本 notebook 纯 PyTorch / CPU，无需 GPU、无需下载模型。

**动手路线**：
1. 内嵌一个单层 mini decoder（带 cache 接口的 causal attention）；
2. 实现两版生成：`generate_naive`（每步全序列前向）vs `generate_with_cache`（K/V 逐步 append）；
3. **正确性 assert**：同权重同 seed 下两版 token 序列逐位一致——这是验证一切"精确推理优化"的标准手段；
4. 计时 + FLOPs 估算曲线：亲眼看到 O(T²) vs O(T)；
5. 显存记账：GPT-2 / Llama-7B / Llama-70B 的 cache 账单；
6. 手写 int8 对称量化，测误差、看 logits 偏移分布——理解"有损优化为什么要重新评测"；
7. ✏️ 3 道练习 + 📖 参考答案。

In [ ]:
import math, time, copy
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

# ---- 单层 mini decoder（精简自模块 02/03，attention 增加了 kv_cache 接口）----
VOCAB, D, N_HEAD, MAX_LEN = 128, 64, 4, 512
DH = D // N_HEAD

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv  = nn.Linear(D, 3 * D, bias=False)
        self.proj = nn.Linear(D, D, bias=False)

    def forward(self, x, kv_cache=None):
        # x: (B, T_new, D); kv_cache: None 或 (k, v)，各为 (B, H, T_past, DH)
        B, T, _ = x.shape
        q, k, v = self.qkv(x).split(D, dim=2)
        q = q.view(B, T, N_HEAD, DH).transpose(1, 2)   # (B, H, T_new, DH)
        k = k.view(B, T, N_HEAD, DH).transpose(1, 2)
        v = v.view(B, T, N_HEAD, DH).transpose(1, 2)
        if kv_cache is not None:                        # 历史 K/V 直接复用，零重算
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        Tq, Tk = q.size(2), k.size(2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(DH)
        # causal mask：query i（全局位置 Tk-Tq+i）只看 key j ≤ 自己
        mask = torch.tril(torch.ones(Tq, Tk, dtype=torch.bool), diagonal=Tk - Tq)
        att = att.masked_fill(~mask, float('-inf'))
        y = att.softmax(dim=-1) @ v                     # (B, H, Tq, DH)
        y = y.transpose(1, 2).reshape(B, Tq, D)
        return self.proj(y), (k, v)

class MiniLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok  = nn.Embedding(VOCAB, D)
        self.pos  = nn.Embedding(MAX_LEN, D)
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.attn = CausalSelfAttention()
        self.mlp  = nn.Sequential(nn.Linear(D, 4 * D), nn.GELU(), nn.Linear(4 * D, D))
        self.ln_f = nn.LayerNorm(D)
        self.head = nn.Linear(D, VOCAB, bias=False)

    def forward(self, idx, kv_cache=None, pos_offset=0):
        # pos_offset：增量解码时告诉模型"这个新 token 在全局第几个位置"
        B, T = idx.shape
        pos = torch.arange(pos_offset, pos_offset + T)
        x = self.tok(idx) + self.pos(pos)
        a, new_cache = self.attn(self.ln1(x), kv_cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return self.head(self.ln_f(x)), new_cache

model = MiniLM().eval()
print('参数量:', sum(p.numel() for p in model.parameters()))

## 1. 两版生成：朴素 vs KV cache

- **`generate_naive`**：每步把*整个序列*重新前向一遍。第 $t$ 步重算 $t$ 个 token 的 Q/K/V，总计 $O(T^2)$。
- **`generate_with_cache`**：先对 prompt 做一次 **prefill** 拿到初始 cache；之后每步只喂 1 个新 token（注意传 `pos_offset`，否则位置编码错位），K/V 逐步 append。每步对新 token 是 $O(1)$ 次 token 级前向，总计 $O(T)$。

两版都用 greedy（argmax）解码——确定性，便于逐位比对。

In [ ]:
@torch.no_grad()
def generate_naive(model, prompt, n_new):
    # 每步全序列前向：正确但 O(T^2)
    idx = prompt.clone()
    for _ in range(n_new):
        logits, _ = model(idx)                              # 整个序列重算！
        nxt = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        idx = torch.cat([idx, nxt], dim=1)
    return idx

@torch.no_grad()
def generate_with_cache(model, prompt, n_new):
    # prefill：prompt 一次前向，建立 cache（占据位置 0..Tp-1）
    Tp = prompt.size(1)
    logits, cache = model(prompt)
    nxt = logits[:, -1, :].argmax(dim=-1, keepdim=True)
    out = [prompt, nxt]
    # decode：每步只算 1 个新 token，对全历史 cache 做注意力
    for step in range(1, n_new):
        logits, cache = model(nxt, kv_cache=cache, pos_offset=Tp + step - 1)
        nxt = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        out.append(nxt)
    return torch.cat(out, dim=1)

print('两版生成函数就绪')

In [ ]:
# ---- 正确性 assert：精确优化的黄金标准 = 输出逐位一致 ----
prompt = torch.randint(0, VOCAB, (1, 8))

seq_naive = generate_naive(model, prompt, n_new=48)
seq_cache = generate_with_cache(model, prompt, n_new=48)

assert seq_naive.shape == seq_cache.shape == (1, 8 + 48)
assert torch.equal(seq_naive, seq_cache), '两版输出不一致——cache 实现有 bug！'

# logits 级别也对一下：对最终序列做一次全量前向，最后一位 logits 应与增量路径几乎相同
full_logits, _ = model(seq_cache[:, :-1])
inc_logits, _  = model(seq_cache[:, -2:-1],
                       kv_cache=model(seq_cache[:, :-2])[1],
                       pos_offset=seq_cache.size(1) - 2)
diff = (full_logits[:, -1] - inc_logits[:, -1]).abs().max().item()
print(f'token 序列逐位一致 ✓   logits 最大偏差 = {diff:.2e}（仅浮点 reduction 噪声）')

## 2. 计时与 FLOPs：亲眼看到 O(T²) vs O(T)

生成 $T=256$ 个 token，对比墙钟时间；再按下式估算**每步 FLOPs**（数量级估算，乘加记 2 FLOPs）：

- 线性层（qkv/proj/mlp/head）每 token ≈ $2(3D^2 + D^2 + 8D^2 + D\cdot V)$；
- attention 每个 query 对 $t$ 个 key ≈ $4tD$（$QK^\top$ 与 $\cdot V$ 各一半）。

朴素版第 $t$ 步要对全部 $t$ 个 token 付线性层成本（且 attention 是 $t^2$ 级），cache 版每步只付 1 个 token 的成本。

In [ ]:
T_GEN = 256
prompt = torch.randint(0, VOCAB, (1, 8))

t0 = time.perf_counter(); _ = generate_naive(model, prompt, T_GEN);      t_naive = time.perf_counter() - t0
t0 = time.perf_counter(); _ = generate_with_cache(model, prompt, T_GEN); t_cache = time.perf_counter() - t0
print(f'墙钟时间  naive: {t_naive:.2f}s   cache: {t_cache:.2f}s   加速 {t_naive/t_cache:.1f}x')

LIN_PER_TOK = 2 * (3*D*D + D*D + 8*D*D + D*VOCAB)   # 每 token 线性层 FLOPs

def flops_step_naive(t):   # 第 t 步：t 个 token 全部重算
    return t * LIN_PER_TOK + sum(4 * i * D for i in range(1, t + 1))

def flops_step_cache(t):   # 第 t 步：只算 1 个新 token，q 对 t 个 key 注意力
    return LIN_PER_TOK + 4 * t * D

steps = list(range(1, T_GEN + 1))
fn = [flops_step_naive(t) for t in steps]
fc = [flops_step_cache(t) for t in steps]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(steps, fn, label='naive (per-step)'); ax[0].plot(steps, fc, label='cache (per-step)')
ax[0].set_xlabel('decode step t'); ax[0].set_ylabel('FLOPs / step'); ax[0].set_yscale('log')
ax[0].set_title('per-step FLOPs'); ax[0].legend()
import numpy as np
ax[1].plot(steps, np.cumsum(fn), label='naive  ~O(T^2)'); ax[1].plot(steps, np.cumsum(fc), label='cache  ~O(T)')
ax[1].set_xlabel('decode step t'); ax[1].set_ylabel('cumulative FLOPs')
ax[1].set_title('cumulative FLOPs'); ax[1].legend()
plt.tight_layout(); plt.show()
print(f'总 FLOPs 比值 naive/cache = {sum(fn)/sum(fc):.1f}x')

## 3. 显存记账：cache 的账单

$$\text{cache bytes} = \underbrace{2}_{K,V} \times L \times H_{kv} \times d_{head} \times T \times \text{bytes(dtype)}$$

权重是常数项；cache 随 $T\times \text{batch}$ 线性增长——长上下文吃显存的是它。下表先用内联算式感受量级（练习 2 你将把它写成函数）。

In [ ]:
GiB = 2**30
configs = [
    # (名称, L, H_kv, d_head, T, dtype_bytes)
    ('GPT-2 small (fp32, 1K)',          12, 12,  64, 1024, 4),
    ('Llama-7B   (fp16, 4K)',           32, 32, 128, 4096, 2),
    ('Llama-70B  MHA 假想 (fp16, 4K)',  80, 64, 128, 4096, 2),
    ('Llama-70B  GQA 实际 (fp16, 4K)',  80,  8, 128, 4096, 2),   # 8 个 KV 头 → 模块 07
]
print(f'{"配置":<34}{"单序列 cache":>14}{"batch=16":>12}')
for name, L, H, dh, T, b in configs:
    n = 2 * L * H * dh * T * b
    print(f'{name:<34}{n/GiB:>11.2f} GiB{16*n/GiB:>9.1f} GiB')
print()
print('对照：7B 权重 fp16 ≈ 14 GB（固定）；70B GQA 把 H_kv 从 64 砍到 8，cache 直接 /8')

## 4. 手写 int8 对称量化

per-tensor 对称量化：整个张量共用一个 scale，

$$s = \frac{\max|W|}{127},\qquad W_{int8} = \mathrm{round}(W/s),\qquad \hat W = s\cdot W_{int8}$$

误差来自 round 舍入（量级 $s/2$）。先量化一个小 Linear 层测**相对误差**与**余弦相似度**；
再把整个 mini 模型量化，看 **logits 偏移分布**——量化是*有损*优化，输出分布会系统性移动，这正是它需要重新评测的原因。

In [ ]:
def quantize(W):
    # per-tensor 对称量化: W ≈ scale * W_int8
    scale = W.abs().max() / 127.0
    W_int8 = torch.round(W / scale).clamp(-127, 127).to(torch.int8)
    return W_int8, scale

def dequantize(W_int8, scale):
    return W_int8.to(torch.float32) * scale

# 对 mlp 的第一个 Linear 层做量化前向对比
lin = model.mlp[0]
W = lin.weight.data
W_hat = dequantize(*quantize(W))

rel_err = (W - W_hat).norm() / W.norm()
cos     = torch.nn.functional.cosine_similarity(W.flatten(), W_hat.flatten(), dim=0)
print(f'权重相对误差 = {rel_err.item():.4f}   余弦相似度 = {cos.item():.6f}')

x = torch.randn(32, D)
y_fp = x @ W.T
y_q  = x @ W_hat.T
out_err = (y_fp - y_q).norm() / y_fp.norm()
print(f'前向输出相对误差 = {out_err.item():.4f}（显存省一半，误差 <1% 量级）')

In [ ]:
# ---- 整模型量化：logits 偏移分布 + greedy 一致率 ----
model_q = copy.deepcopy(model)
for m in model_q.modules():
    if isinstance(m, nn.Linear):
        m.weight.data = dequantize(*quantize(m.weight.data))

batch = torch.randint(0, VOCAB, (16, 64))
with torch.no_grad():
    z_fp, _ = model(batch)
    z_q,  _ = model_q(batch)

shift = (z_q - z_fp).flatten()
agree = (z_q.argmax(-1) == z_fp.argmax(-1)).float().mean()

plt.figure(figsize=(6, 3.5))
plt.hist(shift.numpy(), bins=80)
plt.xlabel('logit shift (int8 - fp32)'); plt.ylabel('count')
plt.title(f'logits shift distribution | top-1 agreement = {agree:.1%}')
plt.tight_layout(); plt.show()

print(f'logits 偏移: max|Δ| = {shift.abs().max():.4f}, std = {shift.std():.4f}')
print(f'greedy top-1 一致率 = {agree:.1%}  ——  注意：不是 100%！')
print('对比第 1 节：KV cache 是精确优化（逐位一致）；量化是有损优化（分布偏移），必须当新模型重测。')

## ✏️ 练习 1：实现 `append_kv`（含滑动窗口截断）

实现 cache 的核心维护操作：把新 token 的 K/V append 到 cache 末尾；若给定 `max_len` 且超长，**丢弃最旧的** token（滑动窗口，长上下文 serving 的常见省显存手段）。

- 输入：`k_cache, v_cache, k_new, v_new` 形状均为 `(B, H, T, DH)`（`k_new/v_new` 的 T 通常为 1）；`max_len` 为 `None` 或正整数。
- 输出：新的 `(k, v)`。时间维是 `dim=2`。
- 提示：`torch.cat` + 负索引切片 `[:, :, -max_len:, :]`，5 行内可完成。

In [ ]:
def append_kv(k_cache, v_cache, k_new, v_new, max_len=None):
    # TODO: 1) 沿时间维 (dim=2) 拼接新 K/V
    # TODO: 2) 若 max_len 不为 None 且超长，只保留最新的 max_len 个位置
    raise NotImplementedError

In [ ]:
# ---- 练习 1 自测 ----
k0 = torch.zeros(1, 2, 0, 3); v0 = torch.zeros(1, 2, 0, 3)
k, v = k0, v0
for i in range(10):   # 依次写入值为 0..9 的 K/V，便于检查"丢的是最旧的"
    ki = torch.full((1, 2, 1, 3), float(i)); vi = torch.full((1, 2, 1, 3), float(i) + 100)
    k, v = append_kv(k, v, ki, vi, max_len=4)

assert k.shape == (1, 2, 4, 3) and v.shape == (1, 2, 4, 3), '窗口截断后形状应为 T=4'
assert torch.equal(k[0, 0, :, 0], torch.tensor([6., 7., 8., 9.])), '应保留最新 4 个、丢最旧的'
assert torch.equal(v[0, 0, :, 0], torch.tensor([106., 107., 108., 109.]))

# 无截断模式
k, v = append_kv(k0, v0, torch.ones(1, 2, 5, 3), torch.ones(1, 2, 5, 3))
k, v = append_kv(k, v, torch.ones(1, 2, 1, 3) * 2, torch.ones(1, 2, 1, 3) * 2)
assert k.shape == (1, 2, 6, 3), 'max_len=None 时不截断'
assert k[0, 0, -1, 0].item() == 2.0, '新 K 应在末尾'

# 边界：恰好等于 max_len 时不截断
k, v = append_kv(k0, v0, torch.ones(1, 2, 4, 3), torch.ones(1, 2, 4, 3), max_len=4)
assert k.shape == (1, 2, 4, 3)
print('✅ 练习 1 通过')

## ✏️ 练习 2：实现 `kv_cache_bytes`

把第 3 节的公式写成函数：`kv_cache_bytes(L, H, dh, T, dtype_bytes)`，返回单条序列 KV cache 的**精确字节数**（int）。

- 公式：$2 \times L \times H \times d_{head} \times T \times \text{bytes}$；
- 提示：1 行 return 即可；用它验证 Llama-7B@4K fp16 恰好是 $2^{31}$ B = 2 GiB。

In [ ]:
def kv_cache_bytes(L, H, dh, T, dtype_bytes):
    # TODO: 返回 2 * L * H * dh * T * dtype_bytes（精确整数）
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测 ----
# Llama-7B: L=32, H=32, dh=128, T=4096, fp16(2B) → 恰好 2 GiB
b7 = kv_cache_bytes(32, 32, 128, 4096, 2)
assert b7 == 2_147_483_648, f'Llama-7B@4K 应为 2147483648 B，得到 {b7}'
assert b7 == 2 * 2**30, '即恰好 2 GiB'

# GPT-2 small: L=12, H=12, dh=64, T=1024, fp32(4B)
assert kv_cache_bytes(12, 12, 64, 1024, 4) == 75_497_472   # ≈ 72 MiB

# 线性性：T 翻倍 → 字节数翻倍；dtype 减半 → 字节数减半
assert kv_cache_bytes(32, 32, 128, 8192, 2) == 2 * b7
assert kv_cache_bytes(32, 32, 128, 4096, 1) == b7 // 2
print('✅ 练习 2 通过')

## ✏️ 练习 3：per-channel 对称量化

per-tensor 量化的弱点：**一个动态范围巨大的通道会污染全局 scale**，让其他通道的有效位宽骤降（LLM.int8 处理的 outlier 问题正是它的激活版）。
实现 `quantize_per_channel(W)`：对 2D 权重 `W (out, in)` 的**每一行**单独计算 scale。

- 返回 `(W_int8, scale)`，其中 `scale` 形状为 `(out, 1)`，满足 `W ≈ W_int8 * scale`；
- 提示：`W.abs().max(dim=1, keepdim=True).values / 127.0`，然后同 per-tensor 一样 round + clamp；
- 自测会在一个"各行尺度相差 5 个数量级"的异方差矩阵上，验证 per-channel 重建误差 **严格小于** per-tensor。

In [ ]:
def quantize_per_channel(W):
    # TODO: 1) 每行一个 scale，形状 (out, 1)
    # TODO: 2) round + clamp 到 [-127, 127]，转 int8
    # TODO: 3) 返回 (W_int8, scale)
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测 ----
torch.manual_seed(0)
# 构造异方差矩阵：各行 std 横跨 1e-3 ~ 1e2
row_scales = torch.tensor([1e-3, 1e-2, 1e-1, 1.0, 10., 100., 0.5, 5.0]).unsqueeze(1)
W_het = torch.randn(8, 64) * row_scales

Wq_pc, s_pc = quantize_per_channel(W_het)
assert Wq_pc.dtype == torch.int8 and s_pc.shape == (8, 1)
W_hat_pc = Wq_pc.to(torch.float32) * s_pc

W_hat_pt = dequantize(*quantize(W_het))             # 第 4 节的 per-tensor 版
err_pc = (W_het - W_hat_pc).norm() / W_het.norm()
err_pt = (W_het - W_hat_pt).norm() / W_het.norm()
print(f'相对误差  per-channel: {err_pc:.2e}   per-tensor: {err_pt:.2e}')

assert err_pc < err_pt, 'per-channel 应严格优于 per-tensor'
assert err_pc < 0.01, 'per-channel 在该矩阵上误差应 <1%'
# 小尺度行在 per-tensor 下被全局 scale 抹平，per-channel 下应保真
r0_pt = (W_het[0] - W_hat_pt[0]).norm() / W_het[0].norm()
r0_pc = (W_het[0] - W_hat_pc[0]).norm() / W_het[0].norm()
assert r0_pc < r0_pt, '最小尺度行的误差应被 per-channel 大幅改善'
print('✅ 练习 3 通过')

## 📖 参考答案

先自己做，再对照。

In [ ]:
# 参考答案 · 练习 1（先自己做，再对照）
def append_kv(k_cache, v_cache, k_new, v_new, max_len=None):
    k = torch.cat([k_cache, k_new], dim=2)
    v = torch.cat([v_cache, v_new], dim=2)
    if max_len is not None and k.size(2) > max_len:
        k = k[:, :, -max_len:, :]
        v = v[:, :, -max_len:, :]
    return k, v

In [ ]:
# 参考答案 · 练习 2（先自己做，再对照）
def kv_cache_bytes(L, H, dh, T, dtype_bytes):
    return 2 * L * H * dh * T * dtype_bytes

In [ ]:
# 参考答案 · 练习 3（先自己做，再对照）
def quantize_per_channel(W):
    scale = W.abs().max(dim=1, keepdim=True).values / 127.0
    W_int8 = torch.round(W / scale).clamp(-127, 127).to(torch.int8)
    return W_int8, scale

## 小结

| 你做了什么 | 对应的工业实践 |
|---|---|
| `generate_with_cache` + 逐位一致 assert | 一切推理引擎的 KV cache；精确优化的数值等价性测试 |
| FLOPs 曲线 O(T²) → O(T) | 为什么没有 cache 就没有可用的 LLM serving |
| `kv_cache_bytes` 记账 | 容量规划：一张卡能 serve 多少并发 × 多长上下文 |
| 滑动窗口 `append_kv` | cache 驱逐 / 长上下文省显存（有损，需评测验证） |
| int8 量化 + logits 偏移直方图 | LLM.int8 / GPTQ 的最简内核；"量化模型 = 新模型，重新评测" |
| per-channel scale | 对抗 outlier 通道——LLM.int8 混合精度分解的同一直觉 |

**评测者备忘**：KV cache / FlashAttention / PagedAttention 不改变输出分布，验证等价性即可；量化与 cache 驱逐改变分布，能力画像（尤其长尾：推理、代码、长上下文检索）必须重测。

→ **模块 07 · 现代架构**：RoPE 如何编码位置、GQA 如何把本章公式里的 $H_{kv}$ 因子砍掉一个数量级、MoE 与 SSM 又如何改写计算/显存的账本。

---
## 🎯 真实数据胶囊题：真实模型的 KV cache 显存与 GQA 节省

用真实 Pythia 配置算 KV cache 显存（2·L·S·h·bytes），并算若改用 GQA（多 query 头共享 g 组 KV）能省多少。

> 本题为本模块新增的**真实数据**练习：自包含，直接用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, numpy as np
CACHE=os.path.expanduser("~/.llm_internals_data"); os.makedirs(CACHE,exist_ok=True)
def load_cfg(model, url):
    p=os.path.join(CACHE,f"{model}.json")
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    c=json.load(open(p)); g=lambda *k: next(c[x] for x in k if x in c); h=g("hidden_size","n_embd")
    return dict(L=g("num_hidden_layers","n_layer"),h=h,heads=g("num_attention_heads","n_head"),
                V=g("vocab_size"),I=c.get("intermediate_size",4*h))
PYTHIA={m:f"https://huggingface.co/EleutherAI/pythia-{m}/resolve/main/config.json"
        for m in ["160m","410m","1.4b","2.8b","6.9b","12b"]}

cfg=load_cfg("pythia-6.9b", PYTHIA["6.9b"])
GB=1024**3
print(f"pythia-6.9b: L={cfg['L']} h={cfg['h']} heads={cfg['heads']}")

**练习**：实现 `kv_gb(cfg,S,batch,bytes)` 和 `gqa_savings(cfg, n_kv_groups)`：MHA 的 KV 正比于 heads；GQA 用 `n_kv_groups` 组 KV（<heads），KV 显存按 `n_kv_groups/heads` 缩小。返回节省比例。

In [ ]:
def kv_gb(cfg, S, batch=1, bytes=2):
    # TODO: 2*L*S*h*bytes*batch / GB
    raise NotImplementedError
def gqa_savings(cfg, n_kv_groups):
    # TODO: 1 - n_kv_groups/heads
    raise NotImplementedError


In [ ]:
# 自测
kv = kv_gb(cfg, 8192, 32)
assert kv > 0 and abs(kv_gb(cfg,16384,32)-2*kv)<1e-6, "S 线性"
s8 = gqa_savings(cfg, 8)
assert 0 < s8 < 1 and s8 > gqa_savings(cfg, 16), "组越少省越多"
assert abs(gqa_savings(cfg, cfg["heads"])) < 1e-9, "组数=头数即 MHA，不省"
print(f"6.9b S=8192 batch=32 KV={kv:.1f}GB; GQA(8组)省 {s8:.0%} ✓")


### 📖 参考答案

In [ ]:
def kv_gb(cfg, S, batch=1, bytes=2):
    return 2*cfg["L"]*S*cfg["h"]*bytes*batch/(1024**3)
def gqa_savings(cfg, n_kv_groups):
    return 1 - n_kv_groups/cfg["heads"]
print("✓ GQA 是现代模型为推理 KV 显存做的标配设计")